# Twi ↔ English Data Cleaning Pipeline

This notebook loads, cleans, and prepares the GhanaNLP parallel text datasets for NLLB fine-tuning.

**Datasets:**
- `Ghana-NLP/TWI_ENGLISH_PARALLEL_TEXT` — Twi source → English target (4,322 pairs)
- `Ghana-NLP/ENGLISH_TWI_PARALLEL_TEXT` — English source → Twi target (6,090 pairs)

**Output:** Cleaned, deduplicated, direction-separated datasets ready for training.

## 1. Install Dependencies

In [ ]:
!pip install -q datasets pandas scikit-learn langdetect huggingface_hub

## 2. Load Both Datasets from HuggingFace Hub

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# Load both datasets from HF Hub
print("Loading datasets from HuggingFace Hub...")
ds1 = load_dataset("Ghana-NLP/TWI_ENGLISH_PARALLEL_TEXT")
ds2 = load_dataset("Ghana-NLP/ENGLISH_TWI_PARALLEL_TEXT")

# Convert to pandas and standardize column names
# Dataset 1: text=twi, label=english
df1 = ds1['train'].to_pandas().rename(columns={'text': 'twi', 'label': 'english'})
df1['source_dataset'] = 'TWI_ENGLISH'

# Dataset 2: text=english, label=twi (reversed!)
df2 = ds2['train'].to_pandas().rename(columns={'text': 'english', 'label': 'twi'})
df2['source_dataset'] = 'ENGLISH_TWI'

print(f"Dataset 1 (TWI_ENGLISH): {len(df1):,} pairs")
print(f"Dataset 2 (ENGLISH_TWI): {len(df2):,} pairs")
print(f"Combined raw: {len(df1) + len(df2):,} pairs")

## 3. Explore Raw Data Quality

In [ ]:
# Twi special characters for language detection
TWI_SPECIAL = set('ɛɔŋƐƆŊ')

def has_twi_chars(text):
    """Check if text contains Twi-specific characters."""
    return any(c in TWI_SPECIAL for c in str(text))

def is_pure_ascii(text):
    """Check if text is pure ASCII (likely English, not Twi)."""
    return all(ord(c) < 128 for c in str(text).replace(' ', ''))

# Analyze Dataset 1
print("=" * 60)
print("Dataset 1 (TWI_ENGLISH) Analysis")
print("=" * 60)
print(f"Null values - twi: {df1['twi'].isna().sum()}, english: {df1['english'].isna().sum()}")
print(f"Twi column with Twi chars: {df1['twi'].apply(has_twi_chars).sum()} ({100*df1['twi'].apply(has_twi_chars).mean():.1f}%)")
print(f"Twi column pure ASCII (swapped?): {df1['twi'].apply(is_pure_ascii).sum()} ({100*df1['twi'].apply(is_pure_ascii).mean():.1f}%)")

# Analyze Dataset 2
print("\n" + "=" * 60)
print("Dataset 2 (ENGLISH_TWI) Analysis")
print("=" * 60)
print(f"Null values - twi: {df2['twi'].isna().sum()}, english: {df2['english'].isna().sum()}")
print(f"Twi column with Twi chars: {df2['twi'].apply(has_twi_chars).sum()} ({100*df2['twi'].apply(has_twi_chars).mean():.1f}%)")
print(f"Twi column pure ASCII (swapped?): {df2['twi'].apply(is_pure_ascii).sum()} ({100*df2['twi'].apply(is_pure_ascii).mean():.1f}%)")

# Sample rows
print("\n" + "=" * 60)
print("Sample rows from Dataset 1")
print("=" * 60)
display(df1[['twi', 'english']].head(3))

## 4. Combine Datasets

In [ ]:
# Keep only relevant columns and combine
df1_clean = df1[['twi', 'english', 'source_dataset']].copy()
df2_clean = df2[['twi', 'english', 'source_dataset']].copy()

# Combine both datasets
df = pd.concat([df1_clean, df2_clean], ignore_index=True)
print(f"Combined dataset: {len(df):,} pairs")

## 5. Text Cleaning Pipeline

In [ ]:
def clean_text(text):
    """Clean a single text string."""
    if not isinstance(text, str) or pd.isna(text):
        return None
    
    # Strip whitespace
    text = text.strip()
    
    # Remove leading list numbers (e.g., "1. ", "2. ")
    text = re.sub(r'^\d+\.\s*', '', text)
    
    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)
    
    # Remove excessive punctuation
    text = re.sub(r'([.!?])\1+', r'\1', text)
    
    # Strip again after processing
    text = text.strip()
    
    return text if text else None

# Apply cleaning
print("Cleaning text...")
df['twi'] = df['twi'].apply(clean_text)
df['english'] = df['english'].apply(clean_text)

# Drop rows with null values after cleaning
before = len(df)
df = df.dropna(subset=['twi', 'english'])
print(f"Dropped {before - len(df):,} rows with null values")
print(f"Remaining: {len(df):,} pairs")

## 6. Length Filtering

In [ ]:
# Calculate word counts
df['twi_wc'] = df['twi'].str.split().str.len()
df['eng_wc'] = df['english'].str.split().str.len()

print("Word count statistics BEFORE filtering:")
print(f"Twi: min={df['twi_wc'].min()}, max={df['twi_wc'].max()}, mean={df['twi_wc'].mean():.1f}")
print(f"English: min={df['eng_wc'].min()}, max={df['eng_wc'].max()}, mean={df['eng_wc'].mean():.1f}")

# Filter: minimum 3 words, maximum 150 words per sentence
MIN_WORDS = 3
MAX_WORDS = 150

before = len(df)
df = df[(df['twi_wc'] >= MIN_WORDS) & (df['eng_wc'] >= MIN_WORDS)]
df = df[(df['twi_wc'] <= MAX_WORDS) & (df['eng_wc'] <= MAX_WORDS)]
print(f"\nDropped {before - len(df):,} rows outside word count range [{MIN_WORDS}, {MAX_WORDS}]")
print(f"Remaining: {len(df):,} pairs")

## 7. Direction-Swapped Pair Detection & Removal

Some pairs have English text in the Twi column (and vice versa). We detect and remove these.

In [ ]:
def is_likely_english(text):
    """Detect if text is likely English (pure ASCII, no Twi chars)."""
    if not isinstance(text, str):
        return False
    # If pure ASCII and no Twi special chars, likely English
    return all(ord(c) < 128 for c in text.replace(' ', '')) and not has_twi_chars(text)

def is_likely_twi(text):
    """Detect if text is likely Twi (has Twi special chars)."""
    return has_twi_chars(text)

# Mark rows where Twi column contains English
df['twi_is_english'] = df['twi'].apply(is_likely_english)
df['english_is_twi'] = df['english'].apply(is_likely_twi)

# Count problematic rows
swapped_twi = df['twi_is_english'].sum()
swapped_eng = df['english_is_twi'].sum()
print(f"Rows with English in Twi column: {swapped_twi} ({100*swapped_twi/len(df):.2f}%)")
print(f"Rows with Twi in English column: {swapped_eng} ({100*swapped_eng/len(df):.2f}%)")

# Remove rows where Twi column is pure ASCII (direction-swapped)
before = len(df)
df = df[~df['twi_is_english']]
print(f"\nRemoved {before - len(df):,} direction-swapped pairs")
print(f"Remaining: {len(df):,} pairs")

# Drop temporary columns
df = df.drop(columns=['twi_is_english', 'english_is_twi'])

## 8. Deduplication

In [ ]:
# Deduplicate exact matches
before = len(df)
df = df.drop_duplicates(subset=['twi', 'english'])
print(f"Removed {before - len(df):,} duplicate pairs")
print(f"Remaining: {len(df):,} unique pairs")

## 9. Length Ratio Filtering

Remove pairs with extreme length ratios (likely misaligned or poor translations).

In [ ]:
# Calculate length ratio
df['length_ratio'] = df['twi_wc'] / df['eng_wc'].clip(lower=1)

print("Length ratio statistics:")
print(f"Min: {df['length_ratio'].min():.2f}")
print(f"Max: {df['length_ratio'].max():.2f}")
print(f"Mean: {df['length_ratio'].mean():.2f}")
print(f"Median: {df['length_ratio'].median():.2f}")

# Remove extreme ratios (< 0.3 or > 3.0)
MIN_RATIO = 0.3
MAX_RATIO = 3.0

before = len(df)
df = df[(df['length_ratio'] >= MIN_RATIO) & (df['length_ratio'] <= MAX_RATIO)]
print(f"\nRemoved {before - len(df):,} pairs with extreme length ratios")
print(f"Remaining: {len(df):,} pairs")

## 10. Final Cleanup

In [ ]:
# Drop helper columns
df = df.drop(columns=['twi_wc', 'eng_wc', 'length_ratio'])

# Reset index
df = df.reset_index(drop=True)

print(f"\n{'='*60}")
print("FINAL CLEANED DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total pairs: {len(df):,}")
print(f"From TWI_ENGLISH: {(df['source_dataset'] == 'TWI_ENGLISH').sum():,}")
print(f"From ENGLISH_TWI: {(df['source_dataset'] == 'ENGLISH_TWI').sum():,}")

# Show samples
print("\nSample cleaned pairs:")
display(df[['twi', 'english']].sample(5, random_state=42))

## 11. Direction-Separated Datasets

For best results, we train TWO models — one per direction. Here we separate the data.

In [ ]:
def is_twi_text(text):
    """Determine if text is Twi based on special characters."""
    return any(c in TWI_SPECIAL for c in str(text))

# Identify the "natural" direction of each pair
# If twi column has Twi chars → pair is Twi→English
# If twi column has no Twi chars → pair was direction-swapped or is English→Twi
df['is_twi_source'] = df['twi'].apply(is_twi_text)

# Split by direction
twi_to_en_df = df[df['is_twi_source']].copy()
en_to_twi_df = df[~df['is_twi_source']].copy()

print(f"Twi → English pairs: {len(twi_to_en_df):,}")
print(f"English → Twi pairs: {len(en_to_twi_df):,}")

# For En→Twi training, we need to swap columns
# The model expects: source → target
# For En→Twi: source=english, target=twi
en_to_twi_df = en_to_twi_df.rename(columns={'twi': 'target', 'english': 'source'})
en_to_twi_df = en_to_twi_df.rename(columns={'source': 'english', 'target': 'twi'})

# Verify
print("\nSample Twi→English pair:")
print(f"  Twi: {twi_to_en_df['twi'].iloc[0][:80]}...")
print(f"  Eng: {twi_to_en_df['english'].iloc[0][:80]}...")

## 12. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

def create_splits(data, name):
    """Create 80/10/10 train/val/test splits."""
    train_df, temp_df = train_test_split(data, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    print(f"\n{name}:")
    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    
    return train_df, val_df, test_df

# Combined dataset splits
train_all, val_all, test_all = create_splits(df, "Combined (bidirectional)")

# Direction-separated splits
train_twi_en, val_twi_en, test_twi_en = create_splits(
    twi_to_en_df[['twi', 'english']], "Twi → English"
)

# Note: en_to_twi may have few pairs after filtering, handle gracefully
if len(en_to_twi_df) > 10:
    train_en_twi, val_en_twi, test_en_twi = create_splits(
        en_to_twi_df[['twi', 'english']], "English → Twi"
    )
else:
    print(f"\nEnglish → Twi: Only {len(en_to_twi_df)} pairs - using all for training")
    train_en_twi = en_to_twi_df
    val_en_twi = pd.DataFrame(columns=['twi', 'english'])
    test_en_twi = pd.DataFrame(columns=['twi', 'english'])

## 13. Save Cleaned Datasets

In [ ]:
import os

# Create output directory
OUTPUT_DIR = '/kaggle/working/cleaned_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save combined dataset
df[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/combined_cleaned.csv', index=False)
train_all[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/train_combined.csv', index=False)
val_all[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/val_combined.csv', index=False)
test_all[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/test_combined.csv', index=False)

# Save direction-separated datasets
train_twi_en.to_csv(f'{OUTPUT_DIR}/train_twi_to_en.csv', index=False)
val_twi_en.to_csv(f'{OUTPUT_DIR}/val_twi_to_en.csv', index=False)
test_twi_en.to_csv(f'{OUTPUT_DIR}/test_twi_to_en.csv', index=False)

if len(train_en_twi) > 0:
    train_en_twi.to_csv(f'{OUTPUT_DIR}/train_en_to_twi.csv', index=False)
    val_en_twi.to_csv(f'{OUTPUT_DIR}/val_en_to_twi.csv', index=False)
    test_en_twi.to_csv(f'{OUTPUT_DIR}/test_en_to_twi.csv', index=False)

print(f"\nSaved all files to {OUTPUT_DIR}")
print("Files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f"  {f}: {size:.1f} KB")

## 14. Summary Statistics

In [ ]:
print("="*70)
print("FINAL DATA CLEANING SUMMARY")
print("="*70)
print(f"\nOriginal combined pairs: 10,412")
print(f"After cleaning: {len(df):,}")
print(f"Pairs removed: {10412 - len(df):,} ({100*(10412-len(df))/10412:.1f}%)")
print(f"\n--- Direction Separation ---")
print(f"Twi → English: {len(twi_to_en_df):,} pairs")
print(f"English → Twi: {len(en_to_twi_df):,} pairs")
print(f"\n--- Train/Val/Test Splits (Combined) ---")
print(f"Train: {len(train_all):,} | Val: {len(val_all):,} | Test: {len(test_all):,}")
print(f"\n--- Data Quality Checks ---")
print(f"All Twi texts contain Twi chars: {df['twi'].apply(has_twi_chars).all()}")
print(f"No null values: {df[['twi', 'english']].isna().sum().sum() == 0}")
print(f"No duplicates: {len(df) == len(df.drop_duplicates(subset=['twi', 'english']))}")
print("\n" + "="*70)
print("Data cleaning complete! Ready for NLLB fine-tuning.")
print("="*70)

## 15. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Word count distributions
df['twi_wc'] = df['twi'].str.split().str.len()
df['eng_wc'] = df['english'].str.split().str.len()

axes[0].hist(df['twi_wc'], bins=50, alpha=0.7, label='Twi', color='blue')
axes[0].hist(df['eng_wc'], bins=50, alpha=0.7, label='English', color='orange')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution')
axes[0].legend()

# Length ratio
df['length_ratio'] = df['twi_wc'] / df['eng_wc'].clip(lower=1)
axes[1].hist(df['length_ratio'], bins=50, color='green', alpha=0.7)
axes[1].set_xlabel('Twi/English Word Count Ratio')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Length Ratio Distribution')
axes[1].axvline(x=1.0, color='red', linestyle='--', label='1:1 ratio')
axes[1].legend()

# Dataset source distribution
source_counts = df['source_dataset'].value_counts()
axes[2].pie(source_counts, labels=source_counts.index, autopct='%1.1f%%', 
            colors=['#66b3ff', '#ff9999'])
axes[2].set_title('Data Source Distribution')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/data_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization saved to {OUTPUT_DIR}/data_statistics.png")